# 一个完整的商业方案

## 现在我们把 Day 1 的项目提升到下一个层次

### 商业挑战：

创建一个产品，为公司生成宣传册（Brochure），供潜在客户、投资者和潜在招聘对象使用。

我们会得到公司名称及其主要网站。

请查看本 notebook 末尾的真实商业应用示例。

记住：如果你有问题或想法，我随时在！请务必联系我。

In [ ]:
# 导入
# 如果这些失败，请检查你是否在命令提示符中带有 (llms) 的「已激活」环境中运行

import os
# 导入 json：把 JSON（JavaScript Object Notation，一种常见数据格式）字符串和 Python 字典互转
import json
# 从 dotenv 导入 load_dotenv：把 .env 文件里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 IPython.display 导入展示工具：在 Jupyter 笔记本里漂亮地显示 Markdown/图片等
from IPython.display import Markdown, display, update_display
# 导入本课程的 scraper 辅助函数：抓取网页正文或链接
from scraper import fetch_website_links, fetch_website_contents
# 从 openai 导入 OpenAI 客户端类：用它调用 Chat Completions 等 API（Application Programming Interface）
from openai import OpenAI

In [ ]:
# 初始化与常量

# 加载 .env 文件：把 API Key 等密钥读入进程环境（override=True 表示覆盖已有同名变量）
load_dotenv(override=True)
# 用 os.getenv 读取环境变量里的密钥；找不到时返回 None
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
# 选定本次实验使用的模型名称（model id）
MODEL = 'gpt-5-nano'
# 创建 OpenAI 客户端；不传参时默认读环境变量里的 API Key
openai = OpenAI()

In [ ]:
# 调用抓取函数，拉取网页正文或链接列表
links = fetch_website_links("https://edwarddonner.com")
links

## 第一步：让 GPT-5-nano 判断哪些链接相关

### 调用 gpt-5-nano 读取网页上的链接，并以结构化 JSON 响应。  
它应决定哪些链接相关，并把 "/about" 这类相对链接替换为 "https://company.com/about"。  
我们将使用「one shot prompting」（单样本提示），在提示中提供它应如何响应的示例。

这是 LLM 的绝佳用例，因为它需要细腻的理解。想象一下不用 LLM、靠解析和分析网页来编程实现——那会非常困难！

补充：有一种更高级的技术叫「Structured Outputs」（结构化输出），我们要求模型按规范响应。我们会在第 8 周的自主 Agentic AI 项目中介绍这项技术。

In [ ]:
# 链接筛选用的系统提示词：告诉模型如何从网页链接里挑出相关项
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [ ]:
# 构造用户提示词：把抓到的链接列表填进模板，交给模型筛选
def get_links_user_prompt(url):
    # 用户提示词（user prompt）：本次要模型完成的具体任务与输入内容
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    # 调用抓取函数，拉取网页正文或链接列表
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [ ]:
print(get_links_user_prompt("https://edwarddonner.com"))

In [ ]:
# 让模型从页面链接里筛选「对公司宣传册有用」的链接，并解析为 JSON
def select_relevant_links(url):
    # 调用 chat.completions.create：向大模型发一次对话请求并拿回复（同步、等全部生成完）
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    # 从响应里取出第一条候选的 message.content（模型生成的文本）
    result = response.choices[0].message.content
    # json.loads：把 JSON 字符串解析成 Python 字典/列表
    links = json.loads(result)
    return links
    

In [ ]:
select_relevant_links("https://edwarddonner.com")

In [ ]:
# 让模型从页面链接里筛选「对公司宣传册有用」的链接，并解析为 JSON
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    # 调用 chat.completions.create：向大模型发一次对话请求并拿回复（同步、等全部生成完）
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    # 从响应里取出第一条候选的 message.content（模型生成的文本）
    result = response.choices[0].message.content
    # json.loads：把 JSON 字符串解析成 Python 字典/列表
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [ ]:
select_relevant_links("https://edwarddonner.com")

In [ ]:
select_relevant_links("https://huggingface.co")

## 第二步：制作宣传册！

把所有细节组装成另一个发给 GPT-5-nano 的提示

In [ ]:
# 抓取首页正文，再抓取模型挑中的相关页面，拼成一份完整素材
def fetch_page_and_all_relevant_links(url):
    # 调用抓取函数，拉取网页正文或链接列表
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    # 遍历（for 循环）：逐个处理序列里的每一项
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        # 调用抓取函数，拉取网页正文或链接列表
        result += fetch_website_contents(link["url"])
    return result

In [ ]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

In [ ]:
# 宣传册生成用的系统提示词：规定输出风格与结构
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# 或者取消下面几行的注释以获得更幽默的宣传册——这展示了融入「语气」有多容易：

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [ ]:
# 构造生成宣传册的用户提示词（公司名 + 页面内容）
def get_brochure_user_prompt(company_name, url):
    # 用户提示词（user prompt）：本次要模型完成的具体任务与输入内容
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [ ]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

In [ ]:
# 一次性生成完整宣传册（等模型全部输出后再显示）
def create_brochure(company_name, url):
    # 调用 chat.completions.create：向大模型发一次对话请求并拿回复（同步、等全部生成完）
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    # 从响应里取出第一条候选的 message.content（模型生成的文本）
    result = response.choices[0].message.content
    # 用 Markdown 在笔记本中渲染格式化文本（display_id 方便后续原地刷新）
    display(Markdown(result))

In [ ]:
create_brochure("HuggingFace", "https://huggingface.co")

## 最后——一个小改进

只需稍作调整，我们就能让结果从 OpenAI 流式返回，
带有熟悉的打字机动画效果

In [ ]:
# 流式生成宣传册：边生成边刷新 Markdown 显示
def stream_brochure(company_name, url):
    # 开启 stream=True 流式输出：模型一边生成，一边把增量文本推过来，体验更像「打字」
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    # 用 Markdown 在笔记本中渲染格式化文本（display_id 方便后续原地刷新）
    display_handle = display(Markdown(""), display_id=True)
    # 按流式 chunk（数据块）拼接文本；delta.content 是本次新增的一小段字
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        # 原地更新同一输出区，实现「打字机」式流式刷新
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
stream_brochure("HuggingFace", "https://huggingface.co")

In [ ]:
# 为 Hugging Face 制作宣传册时，试着把系统提示词改成幽默版本：

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">商业应用</h2>
            <span style="color:#181;">在本练习中，我们扩展了 Day 1 的代码，进行多次 LLM 调用，并生成文档。

这或许是 Agentic AI 设计模式的第一个例子，因为我们组合了对 LLM 的多次调用。第 2 周会有更多这方面的内容，然后在第 8 周当我们构建完全自主的智能体方案时，会以更大的力度回归 Agentic AI。

以这种方式生成内容是最常见的用例之一。与摘要一样，它可以应用于任何业务领域。撰写营销内容、根据规格生成产品教程、创建个性化邮件内容，等等。探索如何将内容生成应用到你的业务中，并尝试为自己做一个概念验证原型。看看 community-contributions 文件夹里其他学员做了什么——有这么多有价值的项目——太疯狂了！</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">进入第 2 周之前（那一周超级有趣）</h2>
            <span style="color:#900;">请查看 week1 EXERCISE notebook，完成第 1 周末的挑战。这将为你提供使用前沿 API 的必要练习，并为第 2 周做好充分准备。</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">提醒：3 个实用资源</h2>
            <span style="color:#f71;">1. 课程资源可在<a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">这里</a>获取。<br/>
            2. 我在 LinkedIn 上，<a href="https://www.linkedin.com/in/eddonner/">点这里</a>，我很喜欢和正在学习本课程的人交流！<br/>
            3. 我在尝试 X/Twitter，账号是 <a href="https://x.com/edwarddonner">@edwarddonner<a>，希望大家能教教我怎么玩……  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">最后！我对你有一个特别请求</h2>
            <span style="color:#090;">
                我的编辑告诉我，学员在 Udemy 上给这门课评分会产生巨大影响——这是 Udemy 决定是否向其他人展示它的主要方式之一。如果你能花一分钟评分，我将万分感激！无论如何——如果在任何时候我能帮上忙，请随时通过 ed@edwarddonner.com 联系我。
            </span>
        </td>
    </tr>
</table>